In [1]:
import findspark
import os
#findspark.init() 
SPARK_HOME='/opt/cloudera/parcels/CDH/lib/spark'
# SPARK_HOME='/home/qiany/.conda/envs/py37'
# os.environ['SPARK_HOME'] = '/home/qiany/.conda/envs/py37'
findspark.init(SPARK_HOME)

In [2]:
import time
import math
import copy
import csv
import json
import os
import codecs
import subprocess
#from hdfs import InsecureClient
import numpy as np
#from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, create_map, array_union,flatten,array_sort,coalesce,broadcast,collect_list, collect_set, udf, array_remove, log, lit, first, col, array, sort_array,split, explode, desc, asc, row_number,isnan, when, count
from pyspark.sql.types import *
import rtree
from pyspark.sql import Window
#import igraph
#from igraph import Graph
import geofeather
from pyspark.storagelevel import StorageLevel

In [3]:
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_tiny_test_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1/ALS_L1B_20190410T153919_155000_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213/ALS_L1B_20190410T174554_181213.off

t_whole_program_0 = time.time()

tin_file = input("Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file:")
# tin_file = '/local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off'

print("\n********************")

# get the directory to the TIN file
tin_directory = os.path.dirname(tin_file)
print("tin_directory: ", tin_directory)

# seaice
# seaice_11122025
# seaice_simplify_ele
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test
# ALS_L1B_20190410T174554_181213_small_crop_tiny_test_pers_0.5
# ALS_L1B_20190410T174554_181213_3_crop_pers_0.25
# ALS_L1B_20190410T174554_181213_3_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_050
# ALS_L1B_20190410T174554_181213_3_pers_010
# ALS_L1B_20190410T174554_181213_3_pers_020
# ALS_L1B_20190410T174554_181213_3_pers_040
# ALS_L1B_20190410T174554_181213_3_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_025
# ALS_L1B_20190410T153919_155000_1_pers_025
# ALS_L1B_20190410T174554_181213_pers_025
# ALS_L1B_20190410T174554_181213_3_pers_015
# ALS_L1B_20190410T174554_181213_3_pers_035
# ALS_L1B_20190410T174554_181213_3_pers_045

# ALS_L1B_20190410T153919_155000_1_pers_020
# ALS_L1B_20190410T153919_155000_1_pers_015
# ALS_L1B_20190410T153919_155000_1_pers_010
# ALS_L1B_20190410T153919_155000_1_pers_030
# ALS_L1B_20190410T153919_155000_1_pers_035
# ALS_L1B_20190410T153919_155000_1_pers_040
# ALS_L1B_20190410T153919_155000_1_pers_045
# ALS_L1B_20190410T153919_155000_1_pers_050

# ALS_L1B_20190410T183726_185444_1_pers_020
# ALS_L1B_20190410T183726_185444_1_pers_015
# ALS_L1B_20190410T183726_185444_1_pers_010
# ALS_L1B_20190410T183726_185444_1_pers_030
# ALS_L1B_20190410T183726_185444_1_pers_035
# ALS_L1B_20190410T183726_185444_1_pers_040
# ALS_L1B_20190410T183726_185444_1_pers_045
# ALS_L1B_20190410T183726_185444_1_pers_050

directory = input("The HDFS directory where you want to store DataFrames:")

# simplify_with_order = input("Simplify with order('y' or 'n'):") or "n"
simplify_with_order = "n"

porsist_thre = input("What is the persistence value threshold (m) you want to set:") or "0.25"

round_idx = input("which round of resimplification you want to perform:")

# get the basename to the TIN file
tin_basename = os.path.basename(tin_file) # input_vertices_2.off
print("tin_basename: ", tin_basename)

# get the filename of the TIN file
tin_filename = os.path.splitext(tin_basename)[0] # input_vertices_2
print("tin_filename: ", tin_filename)

# get the type of TIN file: off, tri, etc
tin_extension = os.path.splitext(tin_basename)[1] # .off
print("tin_extension: ", tin_extension)
print("\n********************")

Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file: /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off



********************
tin_directory:  /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1


The HDFS directory where you want to store DataFrames: ALS_L1B_20190410T183726_185444_1_pers_050
What is the persistence value threshold (m) you want to set: 0.50
which round of resimplification you want to perform: 1


tin_basename:  ALS_L1B_20190410T183726_185444_1.off
tin_filename:  ALS_L1B_20190410T183726_185444_1
tin_extension:  .off

********************


In [4]:
# filtra is the order of each vertex, the order is obtained by ranking elevation values of vertices
# filtra = input("Do you have filtration data?") or "yes"
filtra = 'yes'

if filtra.lower() == 'no':
    Basic_Data = input("Do you have basic pts and tri data?")
    
Num_executor = input("spark.executor.instances:") or "32"
Num_core_per_executor = input("spark.executor.cores:") or "3"
Memory_executor = input("spark.executor.memory? Please end with 'g':") or "56g"
MemoryOverhead_executor = input("spark.executor.memoryOverhead? Please end with 'g':") or "24g"

# Num_core_per_driver = Num_core_per_executor
# Memory_driver = Memory_executor
# MemoryOverhead_driver = MemoryOverhead_executor

Num_core_per_driver = '5'
Memory_driver = '56g'
MemoryOverhead_driver = '24g'

Num_shuffle_partitions = input("spark.sql.shuffle.partitions:") or "288"

spark.executor.instances: 
spark.executor.cores: 
spark.executor.memory? Please end with 'g': 
spark.executor.memoryOverhead? Please end with 'g': 
spark.sql.shuffle.partitions: 


In [5]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import geopandas as gpd
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType, MapType
from shapely.geometry import Point
from shapely.geometry import Polygon

from sedona.register import SedonaRegistrator
from sedona.core.SpatialRDD import SpatialRDD, PointRDD, CircleRDD, PolygonRDD, LineStringRDD
from sedona.core.enums import FileDataSplitter
from sedona.utils.adapter import Adapter
from sedona.core.spatialOperator import KNNQuery
from sedona.core.spatialOperator import JoinQuery
from sedona.core.spatialOperator import JoinQueryRaw
from sedona.core.spatialOperator import RangeQuery
from sedona.core.spatialOperator import RangeQueryRaw
from sedona.core.formatMapper.shapefileParser import ShapefileReader
from sedona.core.formatMapper import WkbReader
from sedona.core.formatMapper import WktReader
from sedona.core.formatMapper import GeoJsonReader
from sedona.sql.types import GeometryType
from sedona.core.enums import GridType
from sedona.core.SpatialRDD import RectangleRDD
from sedona.core.enums import IndexType
from sedona.core.geom.envelope import Envelope
from sedona.utils import SedonaKryoRegistrator, KryoSerializer

In [6]:
os.environ['PYSPARK_PYTHON'] = "./environment/bin/python"
#os.environ['PYSPARK_PYTHON'] = "/home/qiany/.conda/envs/py37/bin/python"
os.environ['YARN_CONF_DIR'] = "/opt/cloudera/parcels/CDH/lib/spark/conf/yarn-conf"

In [7]:
'''
spark.executor.cores: # Number of concurrent tasks an executor can run, euqals to the number of cores to use on each executor
spark.executor.instances: # Number of executors for the spark application
spark.executor.memory: # Amount of memory to use for each executor that runs the task
spark.executor.memoryOverhead:
spark.driver.cores: # Number of cores to use for the driver process; the default number is 1
spark.driver.memory: # Amount of memory to use for the driver
spark.driver.maxResultSize: to define the maximum limit of the total size of the serialized result that a driver can store for each Spark collect action
spark.default.parallelism: # Default number of partitions in RDDs returned by transformations like join, reduceByKey, and parallelize when not set by user. It can be set as spark.executor.instances * spark.executor.cores * 2
spark.sql.shuffle.partitions: determine how many partitions are used when data is shuffled between nodes, e.g., joins or aggregations. usually 1~5 times of executor.instances * executor.cores
spark.memory.storageFraction: determines the fraction of the heap space that is allocated to caching RDDs and DataFrames in memory.
spark.kryoserializer.buffer.max: determine the maximum of data that can be serialized at once; this must be larger than any object we attempt to serialize
spark.rpc.message.maxSize: # Maximum message size (in MiB) to allow in "control plane" communication; generally only applies to map output size information sent between executors and the driver. To communicate between the nodes, Spark uses a protocol called RPC (Remote Procedure Call), which sends messages back and forth. The spark.rpc.message.maxSize parameter limits how big these messages can be. 
spark.sql.broadcastTimeout: Spark will wait for this amount of time before giving up on broadcasting a table. Broadcasting can take a long time if the table is large or if there is a shuffle operation before it.
spark.sql.autoBroadcastJoinThreshold: Spark will broadcast a table to all worker nodes when performing a join if its size is less than this value; -1 means disabling broadcasting
'''

date = time.strftime("%m,%d,%Y")
date_name = date.split(',')[0] + date.split(',')[1] + date.split(',')[2]

hour = time.strftime("%H,%M")
hour_name = hour.split(',')[0] + hour.split(',')[1]

# set the Spark app name
spark_app_name = "Seaice_step7_resimplify_r" + round_idx + "_step2_save_graph_and_result_con_" + tin_filename + '_' + date_name + '_' + hour_name
print("spark_app_name:", spark_app_name)

spark = SparkSession \
.builder \
.appName(spark_app_name) \
.master('yarn') \
.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
.config("spark.kryo.registrator", SedonaKryoRegistrator.getName) \
.config('spark.jars','sedona-core-2.4_2.11-1.0.0-incubating.jar,sedona-sql-2.4_2.11-1.0.0-incubating.jar,sedona-python-adapter-2.4_2.11-1.0.0-incubating.jar,sedona-viz-2.4_2.11-1.0.0-incubating.jar,geotools-wrapper-geotools-24.0.jar,graphframes-0.8.0-spark2.4-s_2.11.jar') \
.config('spark.executor.cores', Num_core_per_executor) \
.config('spark.executor.instances', Num_executor) \
.config('spark.executor.memory', Memory_executor) \
.config('spark.executor.memoryOverhead', MemoryOverhead_executor) \
.config('spark.driver.cores', Num_core_per_driver) \
.config('spark.driver.memory', Memory_driver) \
.config('spark.driver.memoryOverhead', MemoryOverhead_driver) \
.config('spark.driver.maxResultSize', '0') \
.config('spark.dynamicAllocation.enabled', 'false') \
.config('spark.network.timeout', '10000001s') \
.config('spark.executor.heartbeatInterval', '10000000s') \
.config('spark.sql.shuffle.partitions', Num_shuffle_partitions) \
.config("spark.default.parallelism", '200') \
.config("spark.kryoserializer.buffer.max", "1024mb") \
.config('spark.rpc.message.maxSize', '256') \
.config("spark.sql.broadcastTimeout", "36000") \
.config("spark.sql.autoBroadcastJoinThreshold", "-1") \
.config("spark.python.profile", "true") \
.config("spark.eventLog.enabled", "true") \
.config('spark.yarn.dist.archives', '/local/data/yuehui/py37.tar.gz#environment') \
.getOrCreate()

spark_app_name: Seaice_step7_resimplify_r1_step2_save_graph_and_result_con_ALS_L1B_20190410T183726_185444_1_04092026_1210


In [8]:
import sys
sys.path.append("/local/data/yuehui/pyspark/Topo_Simplification/code")

In [9]:
import graphframes
from graphframes import GraphFrame
from graphframes import *
from graphframes.lib import Pregel

In [10]:
file_df_ver_order = directory + '/' + tin_filename + '_filtra_pts' + '.parquet'
df_ver_order = spark.read.parquet(file_df_ver_order)
df_ver_order.printSchema()

root
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- ele: float (nullable = true)
 |-- self_index: integer (nullable = true)
 |-- self_order: integer (nullable = true)



In [11]:
file_df_tri_order = directory + '/' + tin_filename + '_filtra_tri' + '.parquet'
df_tri_order = spark.read.parquet(file_df_tri_order)
df_tri_order.printSchema()

root
 |-- tri_order: integer (nullable = true)
 |-- r1: integer (nullable = true)
 |-- r2: integer (nullable = true)
 |-- r3: integer (nullable = true)
 |-- r1_ele: float (nullable = true)
 |-- r2_ele: float (nullable = true)
 |-- r3_ele: float (nullable = true)



In [12]:
# sort the extreme vertices of a triangle in ascending order, e.g., (2,5,3) -> (2,3,5)

'''
def get_tri_array(r1, r2, r3):
# get_multi_pt_index is used to obtain the adjacent vertexes index, including the vertex itself
# pt_list1: partial adjacent vertex indexes of join result 1
# pt_list2: partial adjacent vertex indexes of join result 2
    tri = [r1, r2, r3]    
    tri.sort(reverse=True)
    
    return tri

# convert a function to an udf and determine the return type
# https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.udf.html
get_tri_array_udf = udf(get_tri_array, ArrayType(IntegerType()))
df_tri_order = df_tri_order.withColumn("tri", get_tri_array_udf(df_tri_order.r1, df_tri_order.r2, df_tri_order.r3))
'''

df_tri_order = df_tri_order.withColumn("tri_origin", F.array("r1", "r2", "r3")).withColumn("tri_ele_origin", F.array("r1_ele", "r2_ele", "r3_ele"))
df_tri_order = df_tri_order.withColumn("tri", sort_array("tri_origin", False)).drop("tri_origin") # sort_array("tri", False), False means descending order
df_tri_order = df_tri_order.withColumn("tri_ele", sort_array("tri_ele_origin", False)).drop("tri_ele_origin") # sort_array("tri", False), False means descending order

df_tri_order.printSchema()
df_tri_order.rdd.getNumPartitions()
#df_tri_order.show()

root
 |-- tri_order: integer (nullable = true)
 |-- r1: integer (nullable = true)
 |-- r2: integer (nullable = true)
 |-- r3: integer (nullable = true)
 |-- r1_ele: float (nullable = true)
 |-- r2_ele: float (nullable = true)
 |-- r3_ele: float (nullable = true)
 |-- tri: array (nullable = false)
 |    |-- element: integer (containsNull = true)
 |-- tri_ele: array (nullable = false)
 |    |-- element: float (containsNull = true)



160

#### get df_crit_E_tri

In [13]:
if round_idx == '1':
    file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris' + '.parquet'
else:
    file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r10' + '.parquet'

df_crit_E_Co_tris = spark.read.parquet(file_df_crit_E_Co_tris) 
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### Graph_VE

In [14]:
if round_idx == '1':
    file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate' + '.parquet'
else:
    file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r10' + '.parquet'

df_V1_result_VE_update_final = spark.read.parquet(file_df_V1_result_VE_update_final)
df_V1_result_VE_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [15]:
df_V1_result_VE_update_final_expl = df_V1_result_VE_update_final.select(explode(df_V1_result_VE_update_final.final_V1_paths).alias("V1_path")).filter(F.col("V1_path").isNotNull()).filter(F.size("V1_path") >= 2)
df_V1_result_VE_update_final_expl.printSchema()

root
 |-- V1_path: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### create an arc DataFrame from V1-paths for the updated version

In [16]:
df_graph_V1_arc_resimplify_r1 = (
    df_V1_result_VE_update_final_expl
    # head = all except last
    .withColumn("head", F.expr("slice(V1_path, 1, size(V1_path)-1)"))
    # tail = all except first
    .withColumn("tail", F.expr("slice(V1_path, 2, size(V1_path)-1)"))
    # pair up head[i], tail[i]
    .withColumn("pairs", F.arrays_zip("head", "tail"))
    # one row per (src, dst)
    .select(F.explode("pairs").alias("e"))
    .select(F.col("e.head").alias("src"), F.col("e.tail").alias("dst"))
)

df_graph_V1_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [17]:
# drop duplicates
df_graph_V1_arc_resimplify_r1 = df_graph_V1_arc_resimplify_r1.dropDuplicates(["src", "dst"])
df_graph_V1_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [18]:
t0 = time.time()

file_df_graph_V1_arc_resimplify_r1 = directory + '/' + tin_filename + '_VE_arc_resimplify_r' + round_idx + '.parquet'
# file_df_graph_V1_arc_resimplify_r1 = directory + '/' + tin_filename + '_VE_arc_resimplify_r11' + '.parquet'

df_graph_V1_arc_resimplify_r1.write.parquet(file_df_graph_V1_arc_resimplify_r1) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 14.539846897125244


#### the nodes are the same as that of before simplification

In [19]:
file_df_graph_V1_node = directory + '/' + tin_filename + '_VE_node' + '.parquet'
df_graph_V1_node = spark.read.parquet(file_df_graph_V1_node)
df_graph_V1_node.printSchema()

root
 |-- id: integer (nullable = true)



#### construct a Graph_VE

In [20]:
graph_VE = GraphFrame(df_graph_V1_node, df_graph_V1_arc_resimplify_r1)

In [21]:
# set a checkpoint directory to improve performance
# Checkpointing regularly helps recover from failures, clean shuffle files, shorten the
# lineage of the computation graph, and reduce the complexity of plan optimization.

spark.sparkContext.setCheckpointDir('checkpoints')

In [22]:
t_con_0 = time.time()
# result_con consists of two columns, vertex id, component
result_con_VE = graph_VE.connectedComponents()
result_con_VE.printSchema()

t_con_1 = time.time()
t_con = t_con_1 - t_con_0
print("time cost of connected components:",t_con)

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)

time cost of connected components: 68.9406988620758


#### distinct number of components

In [23]:
result_con_VE_gpy = result_con_VE.groupby('component').agg(collect_list('id').alias('ids'))
result_con_VE_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [24]:
result_con_VE_gpy_filtered = (
    result_con_VE_gpy
    .filter(F.size(F.col("ids")) > 1)
)

result_con_VE_resimplify_r1_final = result_con_VE_gpy_filtered.select(explode("ids").alias("id"), col("component"))
result_con_VE_resimplify_r1_final.printSchema()

# num_comp_distinct = result_con_VE_gpy_filtered.count()
# print("The number of distinct components:", num_comp_distinct)

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [25]:
# # _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
# if round_idx == '1':
#     file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update' + '.parquet'
# else:
#     file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# # file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r10' + '.parquet'
# df_crit_V = spark.read.parquet(file_df_crit_V)
# df_crit_V.printSchema()

# num_df_crit_V = df_crit_V.count()
# print("The number of critical vertices in df_crit_V_update:", num_df_crit_V)

In [26]:
t_0 = time.time()

file_result_con_VE_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_VE_resimplify_r' + round_idx + '.parquet'

# file_result_con_VE_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_VE_resimplify_r11' + '.parquet'
result_con_VE_resimplify_r1_final.write.parquet(file_result_con_VE_resimplify_r1_final)

t_1 = time.time()
print("time cost of connected components:",t_1 - t_0)

time cost of connected components: 4.437035322189331


### Graph_ET

In [27]:
if round_idx == '1':
    file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate' + '.parquet'
else:
    file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r10' + '.parquet'
df_V2_result_ET_update_final = spark.read.parquet(file_df_V2_result_ET_update_final)
df_V2_result_ET_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [28]:
df_V2_result_ET_update_final_expl = df_V2_result_ET_update_final.select(explode(df_V2_result_ET_update_final.final_V2_paths).alias("V2_path"))
df_V2_result_ET_update_final_expl.printSchema()

root
 |-- V2_path: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### create an arc DataFrame from V2-paths for the updated version

In [29]:
df_graph_V2_arc_resimplify_r1 = (
    df_V2_result_ET_update_final_expl
    # head = all except last
    .withColumn("head", F.expr("slice(V2_path, 1, size(V2_path)-1)"))
    # tail = all except first
    .withColumn("tail", F.expr("slice(V2_path, 2, size(V2_path)-1)"))
    # pair up head[i], tail[i]
    .withColumn("pairs", F.arrays_zip("head", "tail"))
    # one row per (src, dst)
    .select(F.explode("pairs").alias("e"))
    .select(F.col("e.head").alias("src"), F.col("e.tail").alias("dst"))
)

df_graph_V2_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [30]:
# drop duplicates
df_graph_V2_arc_resimplify_r1 = df_graph_V2_arc_resimplify_r1.dropDuplicates(["src", "dst"])
df_graph_V2_arc_resimplify_r1.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



In [31]:
t0 = time.time()

file_df_graph_V2_arc_resimplify_r1 = directory + '/' + tin_filename + '_ET_arc_resimplify_r' + round_idx + '.parquet'

# file_df_graph_V2_arc_resimplify_r1 = directory + '/' + tin_filename + '_ET_arc_resimplify_r11' + '.parquet'
df_graph_V2_arc_resimplify_r1.write.parquet(file_df_graph_V2_arc_resimplify_r1) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.7800192832946777


In [32]:
file_df_graph_V2_node = directory + '/' + tin_filename + '_ET_node' + '.parquet'
df_graph_V2_node = spark.read.parquet(file_df_graph_V2_node)
df_graph_V2_node.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



#### construct a Graph_ET

In [33]:
graph_ET = GraphFrame(df_graph_V2_node, df_graph_V2_arc_resimplify_r1)

In [34]:
# set a checkpoint directory to improve performance
# Checkpointing regularly helps recover from failures, clean shuffle files, shorten the
# lineage of the computation graph, and reduce the complexity of plan optimization.

spark.sparkContext.setCheckpointDir('checkpoints')

In [35]:
t_con_0 = time.time()
# result_con consists of two columns, vertex id, component
result_con_ET = graph_ET.connectedComponents()
result_con_ET.printSchema()

t_con_1 = time.time()
t_con = t_con_1 - t_con_0
print("time cost of connected components:",t_con)

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)

time cost of connected components: 61.6066153049469


#### distinct number of components

In [36]:
result_con_ET_gpy = (
    result_con_ET
    .groupBy("component")
    .agg(
        F.collect_list(F.struct("id", "tri")).alias("items")   # keep id-tri paired
        # or use collect_set(...) if duplicates are possible and you want unique (id,tri)
    )
)

result_con_ET_gpy_filtered = result_con_ET_gpy.filter(F.size("items") > 1)
num_comp_distinct_con_ET = result_con_ET_gpy_filtered.count()
print("The number of distinct components:", num_comp_distinct_con_ET)

result_con_ET_resimplify_r1_final = (
    result_con_ET_gpy_filtered
    .select(F.explode("items").alias("x"), F.col("component"))
    .select(F.col("x.id").alias("id"), F.col("x.tri").alias("tri"), F.col("component"))
)

result_con_ET_resimplify_r1_final.printSchema()

The number of distinct components: 331050
root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [37]:
# # _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
# if round_idx == '1':
#     file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update' + '.parquet'
# else:
#     file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# # file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r10' + '.parquet'
# df_crit_T = spark.read.parquet(file_df_crit_T)
# df_crit_T.printSchema()

# num_df_crit_T = df_crit_T.count()
# print("The number of critical triangles in df_crit_T_update:", num_df_crit_T)

In [38]:
t_0 = time.time()

file_result_con_ET_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_ET_resimplify_r' + round_idx + '.parquet'

# file_result_con_ET_resimplify_r1_final = directory + '/' + tin_filename + '_result_con_ET_resimplify_r11' + '.parquet'
result_con_ET_resimplify_r1_final.write.parquet(file_result_con_ET_resimplify_r1_final)

t_1 = time.time()
print("time cost of connected components:",t_1 - t_0)

time cost of connected components: 6.480010747909546


In [39]:
t_whole_program_1 = time.time()
print(f"Total time cost (s): {(t_whole_program_1 - t_whole_program_0)}")
print(f"Total time cost (min): {(t_whole_program_1 - t_whole_program_0) / 60}")

t_step2 = t_whole_program_1 - t_whole_program_0
print(f"Total time cost (min) for step 2: {t_step2 / 60}")

Total time cost (s): 240.12219262123108
Total time cost (min): 4.0020365436871845
Total time cost (min) for step 2: 4.0020365436871845


# Step 3. save potential simplex pairs to remove

### Extract critical vertices, edeges, and triangles

In [40]:
if round_idx == '1':
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update' + '.parquet'
else:
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r10' + '.parquet'
df_crit_E = spark.read.parquet(file_df_crit_E)
df_crit_E.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [41]:
if round_idx == '1':
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update' + '.parquet'
else:
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r10' + '.parquet'
df_crit_T = spark.read.parquet(file_df_crit_T)
df_crit_T.printSchema()

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### Methods to build Graph_VE

In [42]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r11' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



##### 1) get the boundary vertices of saddles

In [43]:
df_crit_E_pts = (df_crit_E.select(col("Saddle_edge"), explode(col("Saddle_edge")).alias("Saddle_bound_pt")))
df_crit_E_pts.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_bound_pt: integer (nullable = true)



##### 2) inner join result_con_VE with the boundary vertices of saddles to identify the saddles that belong to the same component; typically a saddle belongs to 2 components in Graph_VE

In [44]:
# broadcast left join connected components with saddles
result_con_saddle_VE = result_con_VE.join(df_crit_E_pts,result_con_VE.id==df_crit_E_pts.Saddle_bound_pt, "inner")
result_con_saddle_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_bound_pt: integer (nullable = true)



In [45]:
saddles_per_con_VE = result_con_saddle_VE.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('Saddle_bound_pt').alias('multi_Saddles_pts'))
saddles_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [46]:
t0 = time.time()

file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_resimplify_r' + round_idx + '.parquet'

# file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_resimplify_r11' + '.parquet'
saddles_per_con_VE.write.parquet(file_saddles_per_con_VE) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 2.3771371841430664


### 1. get the minimal minimim-saddle pairs that belong to the same component in G_VE

In [47]:
# get the potential saddle to be simplified within each component
# the potential saddle-minimum pair should have the minimal persistence value among all saddle-minimum pairs
def get_mini_saddle_VE(component, multi_Saddles):
    # get_LS is used to obtain the edges and triangles of LS in ascending order
    # component: the id or index of the minimum of this connected component in Graph_VE
    if len(multi_Saddles) > 0:
        # initialize the potential minimal saddle
        mini_saddle = multi_Saddles[0]
        mini_saddle_persist = mini_saddle[0] # mini_saddle is a pair of vertex, and mini_saddle[0] > mini_saddle[1]
        
        for i in range(1, len(multi_Saddles)):
            if multi_Saddles[i][0] < mini_saddle_persist: # multi_Saddles[i][0] is the persist value of multi_Saddles[i]
                mini_saddle = multi_Saddles[i]
                mini_saddle_persist = mini_saddle[0]                
                
        return mini_saddle
    else:
        return None

get_mini_saddle_VE_udf = udf(get_mini_saddle_VE, ArrayType(IntegerType()))

result_con_saddle_VE_mini = saddles_per_con_VE.withColumn("mini_saddle", get_mini_saddle_VE_udf(saddles_per_con_VE.component, saddles_per_con_VE.multi_Saddles))
result_con_saddle_VE_mini.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- mini_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [48]:
t0 = time.time()

file_result_con_saddle_VE_mini = directory + '/' + tin_filename + '_result_con_saddle_VE_mini_resimplify_r' + round_idx + '.parquet'

# file_result_con_saddle_VE_mini = directory + '/' + tin_filename + '_result_con_saddle_VE_mini_resimplify_r11' + '.parquet'
result_con_saddle_VE_mini.write.parquet(file_result_con_saddle_VE_mini) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 2.9302384853363037


### Methods to build Graph_ET

#### get result_con_ET

In [49]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r11' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



##### 1)  get the co-boundary triangles of saddles

In [50]:
if round_idx == '1':
    file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris' + '.parquet'
else:
    file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_E_Co_tris_origin = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r10' + '.parquet'
df_crit_E_Co_tris_origin = spark.read.parquet(file_df_crit_E_Co_tris_origin)
df_crit_E_Co_tris_origin.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [51]:
df_crit_E_Co_tris = df_crit_E_Co_tris_origin.join(df_crit_E, df_crit_E_Co_tris_origin.Saddle_edge==df_crit_E.Saddle_edge, "inner")
df_crit_E_Co_tris = df_crit_E_Co_tris.select(df_crit_E_Co_tris_origin.Saddle_edge, df_crit_E_Co_tris_origin.Saddle_Co_t)
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [52]:
t0 = time.time()

file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r' + round_idx + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r11' + '.parquet'
df_crit_E_Co_tris.write.parquet(file_df_crit_E_Co_tris) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.3776774406433105


##### 2) inner join result_con_ET with the co-boundary triangles of saddles to identify the saddles that belong to the same component; typically a saddle belongs to 2 component in Graph_ET

In [53]:
# broadcast left join connected components with saddles
saddles_per_con_ET_init = result_con_ET.join(df_crit_E_Co_tris,result_con_ET.tri==df_crit_E_Co_tris.Saddle_Co_t, "inner")

saddles_per_con_ET = saddles_per_con_ET_init.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('id').alias('multi_Saddles_co_tri_id'))
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [54]:
t0 = time.time()

file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_resimplify_r' + round_idx + '.parquet'

# file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_resimplify_r11' + '.parquet'
saddles_per_con_ET.write.parquet(file_saddles_per_con_ET) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 3.085631847381592


##### 3) inner join result_con with critical triangles to identify the component id of critical triangle 

In [55]:
df_crit_T = df_crit_T.select("Max_tri")

df_crit_T_con = df_crit_T.join(result_con_ET, df_crit_T.Max_tri==result_con_ET.tri, "left")
df_crit_T_con = df_crit_T_con.select("id", "component", "Max_tri")
df_crit_T_con.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### 4) join saddles_per_con_ET with df_crit_T_con to get the corresponding critical triangle of a component ID

In [56]:
result_con_saddle_ET = saddles_per_con_ET.join(df_crit_T_con, saddles_per_con_ET.component==df_crit_T_con.component, "left")
result_con_saddle_ET = result_con_saddle_ET.select(saddles_per_con_ET.component, "Max_tri", "multi_Saddles")
result_con_saddle_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



### 2. get the minimal saddle-maximum pairs belonging to the same component in G_ET

In [57]:
# get the potential saddle to be simplified within each component
# the potential saddle-maximum pair should have the minimal persistence value among all saddle-maximum pairs
def get_mini_saddle_ET(Max_tri, multi_Saddles):
    # Max_tri: the critical triangle in this component
    # multi_Saddles: the list of saddles in the same componnt as Max_tri
    if len(multi_Saddles) > 0:
        # initialize the potential minimal saddle
        mini_saddle = multi_Saddles[0]
        mini_saddle_persist = mini_saddle[0] # mini_saddle is a pair of vertex, and mini_saddle[0] > mini_saddle[1]
        
        for i in range(1, len(multi_Saddles)):
            if multi_Saddles[i][0] > mini_saddle_persist: # multi_Saddles[i][0] is the persist value of multi_Saddles[i]
                mini_saddle = multi_Saddles[i]
                mini_saddle_persist = mini_saddle[0]                
                
        return mini_saddle
    else:
        return None

get_mini_saddle_ET_udf = udf(get_mini_saddle_ET, ArrayType(IntegerType()))

result_con_saddle_ET_mini = result_con_saddle_ET.withColumn("mini_saddle", get_mini_saddle_ET_udf(result_con_saddle_ET.component, result_con_saddle_ET.multi_Saddles))
result_con_saddle_ET_mini.printSchema()

root
 |-- component: long (nullable = true)
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- mini_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [58]:
t0 = time.time()

file_result_con_saddle_ET_mini = directory + '/' + tin_filename + '_result_con_saddle_ET_mini_resimplify_r' + round_idx + '.parquet'

# file_result_con_saddle_ET_mini = directory + '/' + tin_filename + '_result_con_saddle_ET_mini_resimplify_r11' + '.parquet'
result_con_saddle_ET_mini.write.parquet(file_result_con_saddle_ET_mini) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 4.444810390472412


### 3. make each saddle as a center, extract the minima and maxima connected to this saddle

In [59]:
con_saddle_1V1_VE = saddles_per_con_VE.select(col("component"), explode(saddles_per_con_VE.multi_Saddles).alias("saddle_single_VE"))

Con_OF_saddle_VE = con_saddle_1V1_VE.groupby('saddle_single_VE').agg(collect_list('component').alias('multi_components_VE'))
Con_OF_saddle_VE.printSchema()

root
 |-- saddle_single_VE: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)



In [60]:
con_saddle_1V1_ET = result_con_saddle_ET.select(col("component"), col("Max_tri"), explode(result_con_saddle_ET.multi_Saddles).alias("saddle_single_ET"))

Con_OF_saddle_ET = con_saddle_1V1_ET.groupby('saddle_single_ET').agg(collect_list('component').alias('multi_components_ET'), collect_list('Max_tri').alias('multi_Max_tri'))
Con_OF_saddle_ET.printSchema()

root
 |-- saddle_single_ET: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_components_ET: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



##### join the two sub-components of saddles

In [61]:
DF_con_OF_saddle = Con_OF_saddle_VE.join(Con_OF_saddle_ET, Con_OF_saddle_VE.saddle_single_VE==Con_OF_saddle_ET.saddle_single_ET, "inner")
DF_con_OF_saddle = DF_con_OF_saddle.select(col("saddle_single_VE").alias("saddle"), "multi_components_VE", "multi_components_ET", "multi_Max_tri")
DF_con_OF_saddle.printSchema()

root
 |-- saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_components_ET: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



### 4.1. join the component of minima and the component of saddle to remove saddle-minima pairs

In [62]:
DF_saddle_minima_init = result_con_saddle_VE_mini.join(DF_con_OF_saddle, result_con_saddle_VE_mini.mini_saddle==DF_con_OF_saddle.saddle, "left")
DF_saddle_minima_init = DF_saddle_minima_init.select(DF_con_OF_saddle.saddle.alias("pot_saddle"), result_con_saddle_VE_mini.component.alias("pot_minimum"), "multi_components_VE", "multi_Max_tri")
DF_saddle_minima_init.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_minimum: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [63]:
t0 = time.time()

def hdfs_path_exists(spark, path_str):
    jvm = spark._jvm
    hadoop_conf = spark._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    path = jvm.org.apache.hadoop.fs.Path(path_str)
    return fs.exists(path)

# get the elevation values of each vertex
file_df_ver_order_ele_sort = directory + '/' + tin_filename + '_filtra_pts_sort' + '.parquet'
if hdfs_path_exists(spark, file_df_ver_order_ele_sort):
    df_ver_order_read = spark.read.parquet(file_df_ver_order_ele_sort)
    ver_dict = {row['self_order']: row['ele'] for row in df_ver_order_read.collect()}
else:
    ver_dict = {row['self_order']: row['ele'] for row in df_ver_order.collect()}

# Broadcast it (this is handled differently than a standard closure)
sc = spark.sparkContext
broadcast_ver = sc.broadcast(ver_dict)
# Access the dictionary via .value
lookup = broadcast_ver.value

t1 = time.time()
print("time cost:", t1-t0)

time cost: 76.32789063453674


In [64]:
if simplify_with_order == '1' or simplify_with_order == 'yes' or simplify_with_order == 'y':
    # check if the saddle-minima pair could be contracted
    def contract_VE(pot_saddle, pot_minimum, multi_components_VE, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the saddle-minima pair
        # pot_minimum: the critical vertex to be contracted in the saddle-minima pair
        # multi_components_VE: the other critical vertices that are connected with the pot_saddle
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        
        if pot_saddle == None or pot_minimum == None:
            return
        
        persist_value = pot_saddle[0] - pot_minimum
#         persist_value_thre = 1000
#         if persist_value < persist_value_thre:
#             less_than_thre = True
#         else:
#             less_than_thre = False
        
        less_than_thre = True
        smallest_mini_saddle = True
        smallest_max_saddle = True
        if len(multi_components_VE) > 0:
            for crit_ver in multi_components_VE:
                if pot_saddle[0] - crit_ver < persist_value:
                # if crit_ver < pot_minimum:
                    smallest_mini_saddle = False
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if crit_tri[0] - pot_saddle[0] < persist_value:
                    smallest_max_saddle = False
            
        contract = smallest_mini_saddle and smallest_max_saddle and less_than_thre
        return contract
else:
    print("Simplification according to elevation!")
#     # get the elevation values of each vertex
#     file_df_ver_order_ele_sort = directory + '/' + tin_filename + '_filtra_pts_sort' + '.parquet'
#     if hdfs_path_exists(spark, file_df_ver_order_ele_sort):
#         df_ver_order_read = spark.read.parquet(file_df_ver_order_ele_sort)
#         df_ver_order_col = df_ver_order_read.collect()
#     else:
#         df_ver_order = df_ver_order.sort(col('self_order'), ascending=True)
#         df_ver_order_col = df_ver_order.collect()
#     # check if the saddle-minima pair could be contracted
    
    def contract_VE(pot_saddle, pot_minimum, multi_components_VE, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the saddle-minima pair
        # pot_minimum: the critical vertex to be contracted in the saddle-minima pair
        # multi_components_VE: the other critical vertices that are connected with the pot_saddle
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        
        if pot_saddle == None or pot_minimum == None:
            return
        
        # persist_value = df_ver_order_col[pot_saddle[0]]['ele'] - df_ver_order_col[pot_minimum]['ele']
        persist_value = lookup.get(pot_saddle[0]) - lookup.get(pot_minimum)
        persist_value_thre = float(porsist_thre)
        if persist_value < persist_value_thre:
            less_than_thre = True
        else:
            less_than_thre = False
        
    #     less_than_thre = True
        smallest_mini_saddle = True
        smallest_max_saddle = True
        if len(multi_components_VE) > 0:
            for crit_ver in multi_components_VE:
                if lookup.get(pot_saddle[0]) - lookup.get(crit_ver) < persist_value:
                # if df_ver_order_col[pot_saddle[0]]['ele'] - df_ver_order_col[crit_ver]['ele'] < persist_value:
                # if crit_ver < pot_minimum:
                    smallest_mini_saddle = False
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if lookup.get(crit_tri[0]) - lookup.get(pot_saddle[0]) < persist_value:
                # if df_ver_order_col[crit_tri[0]]['ele'] - df_ver_order_col[pot_saddle[0]]['ele'] < persist_value:
                    smallest_max_saddle = False
            
        saddle_smaller_than_thre = True
#         if df_ver_order_col[pot_saddle[1]]['ele'] >= 0.5:
#             saddle_smaller_than_thre = False
            
        contract = smallest_mini_saddle and smallest_max_saddle and saddle_smaller_than_thre and less_than_thre
        return contract
    
contract_VE_udf = udf(contract_VE, BooleanType())

DF_saddle_minima = DF_saddle_minima_init.withColumn("contract_VE_or_not", contract_VE_udf(DF_saddle_minima_init.pot_saddle, DF_saddle_minima_init.pot_minimum, DF_saddle_minima_init.multi_components_VE, DF_saddle_minima_init.multi_Max_tri))
DF_saddle_minima.printSchema()

Simplification according to elevation!
root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_minimum: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_VE_or_not: boolean (nullable = true)



In [65]:
t0 = time.time()

file_DF_saddle_minima = directory + '/' + tin_filename + '_DF_saddle_minima_resimplify_r' + round_idx + '.parquet'

# file_DF_saddle_minima = directory + '/' + tin_filename + '_DF_saddle_minima_resimplify_r11' + '.parquet'
DF_saddle_minima.write.parquet(file_DF_saddle_minima) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 23.56706953048706


In [66]:
# 4.2. join the component of maxima and the component of saddle to remove maxima-saddle pairs

In [67]:
DF_maxima_saddle_init = result_con_saddle_ET_mini.join(DF_con_OF_saddle, result_con_saddle_ET_mini.mini_saddle==DF_con_OF_saddle.saddle, "left")
DF_maxima_saddle_init = DF_maxima_saddle_init.select(DF_con_OF_saddle.saddle.alias("pot_saddle"), result_con_saddle_ET_mini.Max_tri.alias("pot_maximum"), result_con_saddle_ET_mini.component.alias("pot_maximum_id"), "multi_components_VE", "multi_Max_tri")
DF_maxima_saddle_init.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [68]:
if simplify_with_order == '1' or simplify_with_order == 'yes' or simplify_with_order == 'y':
    # check if the maxima-saddle pair could be contracted
    def contract_ET(pot_saddle, pot_maximum, multi_components_VE, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the maxima-saddle pair
        # pot_maximum: the critical triangle to be contracted in the maxima-saddle pair
        # multi_components_VE: the other critical vertices that are connected with the pot_saddle
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        if pot_saddle == None or pot_maximum == None:
            return
        
        persist_value = pot_maximum[0] - pot_saddle[0]
#         persist_value_thre = 1000
#         if persist_value < persist_value_thre:
#             less_than_thre = True
#         else:
#             less_than_thre = False
            
        less_than_thre = True
        smallest_mini_saddle = True
        smallest_max_saddle = True
        if len(multi_components_VE) > 0:
            for crit_ver in multi_components_VE:
                if pot_saddle[0] - crit_ver < persist_value:
                    smallest_mini_saddle = False
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if crit_tri[0] - pot_saddle[0] < persist_value:
                    smallest_max_saddle = False
            
        contract = smallest_mini_saddle and smallest_max_saddle and less_than_thre
        return contract
else:
    print("Simplification according to elevation!")
    # check if the maxima-saddle pair could be contracted
    def contract_ET(pot_saddle, pot_maximum, multi_components_VE, multi_Max_tri):
        # pot_saddle: the center saddle, which is the saddle to be contracted in the maxima-saddle pair
        # pot_maximum: the critical triangle to be contracted in the maxima-saddle pair
        # multi_components_VE: the other critical vertices that are connected with the pot_saddle
        # multi_Max_tri: the critical triangles that are connected with the pot_saddle
        if pot_saddle == None or pot_maximum == None:
            return
        
        # persist_value = df_ver_order_col[pot_maximum[0]]['ele'] - df_ver_order_col[pot_saddle[0]]['ele']
        persist_value = lookup.get(pot_maximum[0]) - lookup.get(pot_saddle[0])
        persist_value_thre = float(porsist_thre)
        if persist_value < persist_value_thre:
            less_than_thre = True
        else:
            less_than_thre = False
            
    #     less_than_thre = True
        smallest_mini_saddle = True
        smallest_max_saddle = True
        if len(multi_components_VE) > 0:
            for crit_ver in multi_components_VE:
                if lookup.get(pot_saddle[0]) - lookup.get(crit_ver) < persist_value:
                # if df_ver_order_col[pot_saddle[0]]['ele'] - df_ver_order_col[crit_ver]['ele'] < persist_value:
                    smallest_mini_saddle = False
                    
        if len(multi_Max_tri) > 0:
            for crit_tri in multi_Max_tri:
                if lookup.get(crit_tri[0]) - lookup.get(pot_saddle[0]) < persist_value:
                # if df_ver_order_col[crit_tri[0]]['ele'] - df_ver_order_col[pot_saddle[0]]['ele'] < persist_value:
                    smallest_max_saddle = False
            
        contract = smallest_mini_saddle and smallest_max_saddle and less_than_thre
        return contract

contract_ET_udf = udf(contract_ET, BooleanType())

DF_maxima_saddle = DF_maxima_saddle_init.withColumn("contract_ET_or_not", contract_ET_udf(DF_maxima_saddle_init.pot_saddle, DF_maxima_saddle_init.pot_maximum, DF_maxima_saddle_init.multi_components_VE, DF_maxima_saddle_init.multi_Max_tri))
DF_maxima_saddle.printSchema()

Simplification according to elevation!
root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)



In [69]:
t0 = time.time()

file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_resimplify_r' + round_idx + '.parquet'

# file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_resimplify_r11' + '.parquet'
DF_maxima_saddle.write.parquet(file_DF_maxima_saddle) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 23.04979681968689


In [70]:
t_whole_program_2 = time.time()
print(f"Total time cost (s): {(t_whole_program_2 - t_whole_program_1)}")
print(f"Total time cost (min): {(t_whole_program_2 - t_whole_program_1) / 60}")

t_step3 = t_whole_program_2 - t_whole_program_1
print(f"Total time cost (min) for step 3: {t_step3 / 60}")

Total time cost (s): 202.52673435211182
Total time cost (min): 3.375445572535197
Total time cost (min) for step 3: 3.375445572535197


# Step 4. save V1- and V2- paths

### 1. Methods to build Graph_VE

In [71]:
file_df_graph_V1_arc = directory + '/' + tin_filename + '_VE_arc_resimplify_r' + round_idx + '.parquet'

# file_df_graph_V1_arc = directory + '/' + tin_filename + '_VE_arc_resimplify_r11' + '.parquet'
df_graph_V1_arc = spark.read.parquet(file_df_graph_V1_arc)
df_graph_V1_arc.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



#### get result_con of G_VE

In [72]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r11' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema() # have checked that each id is identical in result_con_VE

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [73]:
file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_resimplify_r' + round_idx + '.parquet'

# file_saddles_per_con_VE = directory + '/' + tin_filename + '_saddles_per_con_VE_resimplify_r11' + '.parquet'
saddles_per_con_VE = spark.read.parquet(file_saddles_per_con_VE)
saddles_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### 2. get V1-paths

##### get the extreme vertices of saddles that belong to the same component

In [74]:
saddles_Pts_per_con_VE = saddles_per_con_VE.select("component", "multi_Saddles_pts")
saddles_Pts_per_con_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the directed arcs (VE pairs) that belong to the same component

In [75]:
# left join df_graph_V1_arc with result_con to identify which component that each arc belongs to
df_G_V1_arc_component = df_graph_V1_arc.join(result_con_VE,df_graph_V1_arc.src==result_con_VE.id, "left").drop('id')
df_G_V1_arc_component.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)
 |-- component: long (nullable = true)



##### store each component in the same row of a DataFrame

In [76]:
# convert each arc to a dictionary
df_G_V1_arc_component_dict = df_G_V1_arc_component.withColumn("arcs", create_map(col('src'), col('dst'))).drop('src','dst')

# combine dictionaries belonging to an identical component to one list
df_G_V1_arc_component_dict_gpy = df_G_V1_arc_component_dict.groupBy('component').agg(collect_list('arcs').alias('subgraphs'))

# left join df_G_V1_arc_component_dict_gpy with unique_SdlPts_per_con to obtain SdlPts belonging to each component
df_G_V1_arc_component_dict_gpy = df_G_V1_arc_component_dict_gpy.join(saddles_Pts_per_con_VE,df_G_V1_arc_component_dict_gpy.component==saddles_Pts_per_con_VE.component, "left").select(df_G_V1_arc_component_dict_gpy.component, 'multi_Saddles_pts', 'subgraphs')
df_G_V1_arc_component_dict_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)



In [77]:
import collections
def bfs_12152025(multi_Saddles_pts, subgraphs):
    '''
    multi_Saddles_pts: an array of string or an array of integers
    subgraphs: an array of dictionary, where each key is a string (or integer) and the corresponding value is also a string (or integer)
    '''
    if not multi_Saddles_pts:
        return None
    # Build the adjacency list from the list of edge dictionaries
    adj_list = collections.defaultdict()
    for edge_dict in subgraphs:
        # Assuming each dictionary has only one key-value pair representing an edge
        for source, destination in edge_dict.items():
            adj_list[source] = destination

    V1_paths = []
    for i in range(len(multi_Saddles_pts)):
        node = multi_Saddles_pts[i]
        V1_path_temp = []
        V1_path_temp_set = set()
        
        while node != -1 and node not in V1_path_temp_set:
            V1_path_temp.append(node)
            V1_path_temp_set.add(node)
            next_node = adj_list.get(node, -1)
            node = next_node
        
        V1_paths.append(V1_path_temp)
    
    return V1_paths

bfs_df_udf = udf(bfs_12152025, ArrayType(ArrayType(IntegerType())))

In [78]:
# Use mapPartitions to apply the bfs function to each partition of the RDD
# Each partition corresponds to a single graph that is processed by a single executor
t_V1_0 = time.time()

df_V1_result_VE_12152025 = df_G_V1_arc_component_dict_gpy.withColumn("V1_paths", bfs_df_udf(df_G_V1_arc_component_dict_gpy.multi_Saddles_pts, df_G_V1_arc_component_dict_gpy.subgraphs))
df_V1_result_VE_12152025.printSchema()
# num_df_V1_result_VE = df_V1_result_VE.count()

t_V1_1 = time.time()
print("Time cost to extract V1-paths of Graph_VE:", t_V1_1 - t_V1_0)
# print("The number of V1-paths in Graph_VE:", num_df_V1_result_VE)

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)

Time cost to extract V1-paths of Graph_VE: 0.02259206771850586


In [79]:
t0 = time.time()

file_df_V1_result_VE_12152025 = directory + '/' + tin_filename + '_df_V1_result_VE_resimplify_r' + round_idx + '.parquet'

# file_df_V1_result_VE_12152025 = directory + '/' + tin_filename + '_df_V1_result_VE_resimplify_r11' + '.parquet'
df_V1_result_VE_12152025.write.parquet(file_df_V1_result_VE_12152025) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 3.602820634841919


### 3. Methods to build Graph_ET

In [80]:
file_df_graph_V2_edge_final = directory + '/' + tin_filename + '_ET_arc_resimplify_r' + round_idx + '.parquet'

# file_df_graph_V2_edge_final = directory + '/' + tin_filename + '_ET_arc_resimplify_r11' + '.parquet'
df_graph_V2_edge_final = spark.read.parquet(file_df_graph_V2_edge_final)
df_graph_V2_edge_final.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)



#### get result_con_ET

In [81]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r11' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [82]:
file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_resimplify_r' + round_idx + '.parquet'

# file_saddles_per_con_ET = directory + '/' + tin_filename + '_saddles_per_con_ET_resimplify_r11' + '.parquet'
saddles_per_con_ET = spark.read.parquet(file_saddles_per_con_ET)
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### 4. get V2-paths

##### get the co-boundary triangles of saddles that belong to the same component

In [83]:
saddles_Tri_per_con_ET = saddles_per_con_ET.select("component", "multi_Saddles_co_tri_id")
saddles_Tri_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the ET pairs (arcs of Graph_ET) that belong to the same component

In [84]:
# left join df_graph_V1_arc with result_con to identify which component that each arc belongs to
df_G_V2_arc_component = df_graph_V2_edge_final.join(result_con_ET,df_graph_V2_edge_final.src==result_con_ET.id, "left").drop('id', 'tri')
df_G_V2_arc_component.printSchema()

root
 |-- src: integer (nullable = true)
 |-- dst: integer (nullable = true)
 |-- component: long (nullable = true)



In [85]:
# convert each arc to a dictionary
df_G_V2_arc_component_dict = df_G_V2_arc_component.withColumn("arcs", create_map(col('src'), col('dst'))).drop('src','dst')

# combine dictionaries belonging to an identical component to one list
df_G_V2_arc_component_dict_gpy = df_G_V2_arc_component_dict.groupBy('component').agg(collect_list('arcs').alias('subgraphs'))

# left join df_G_V1_arc_component_dict_gpy with unique_SdlPts_per_con to obtain SdlPts belonging to each component
df_G_V2_arc_component_dict_gpy = df_G_V2_arc_component_dict_gpy.join(saddles_Tri_per_con_ET,df_G_V2_arc_component_dict_gpy.component==saddles_Tri_per_con_ET.component, "left").select(df_G_V2_arc_component_dict_gpy.component, 'multi_Saddles_co_tri_id', 'subgraphs')
df_G_V2_arc_component_dict_gpy.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)



In [86]:
# Use mapPartitions to apply the bfs function to each partition of the RDD
# Each partition corresponds to a single graph that is processed by a single executor
t_V1_0 = time.time()

df_V2_result_ET = df_G_V2_arc_component_dict_gpy.withColumn("V2_paths", bfs_df_udf(df_G_V2_arc_component_dict_gpy.multi_Saddles_co_tri_id, df_G_V2_arc_component_dict_gpy.subgraphs))
df_V2_result_ET.printSchema()
# num_df_V2_result_ET = df_V2_result_ET.count()

t_V1_1 = time.time()
print("Time cost to extract V1-paths of Graph_VE:", t_V1_1 - t_V1_0)
# print("The number of V1-paths in Graph_VE:", num_df_V2_result_ET)

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)

Time cost to extract V1-paths of Graph_VE: 0.0036089420318603516


In [87]:
t0 = time.time()

file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_resimplify_r' + round_idx + '.parquet'

# file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_resimplify_r11' + '.parquet'
df_V2_result_ET.write.parquet(file_df_V2_result_ET) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 4.082407474517822


In [88]:
t_whole_program_3 = time.time()
print(f"Total time cost (s): {(t_whole_program_3 - t_whole_program_2)}")
print(f"Total time cost (min): {(t_whole_program_3 - t_whole_program_2) / 60}")

t_step4 = t_whole_program_3 - t_whole_program_2
print(f"Total time cost (min) for step 4: {t_step4 / 60}")

Total time cost (s): 9.479774236679077
Total time cost (min): 0.15799623727798462
Total time cost (min) for step 4: 0.15799623727798462


# Step 5. reverse and eliminate V1- and V2- paths

In [89]:
file_DF_saddle_minima = directory + '/' + tin_filename + '_DF_saddle_minima_resimplify_r' + round_idx + '.parquet'

# file_DF_saddle_minima = directory + '/' + tin_filename + '_DF_saddle_minima_resimplify_r11' + '.parquet'
DF_saddle_minima = spark.read.parquet(file_DF_saddle_minima)
DF_saddle_minima.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_minimum: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_VE_or_not: boolean (nullable = true)



In [90]:
file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_resimplify_r' + round_idx + '.parquet'

# file_DF_maxima_saddle = directory + '/' + tin_filename + '_DF_maxima_saddle_resimplify_r11' + '.parquet'
DF_maxima_saddle = spark.read.parquet(file_DF_maxima_saddle)
DF_maxima_saddle.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)



In [91]:
file_df_V1_result_VE = directory + '/' + tin_filename + '_df_V1_result_VE_resimplify_r' + round_idx + '.parquet'

# file_df_V1_result_VE = directory + '/' + tin_filename + '_df_V1_result_VE_resimplify_r11' + '.parquet'
df_V1_result_VE = spark.read.parquet(file_df_V1_result_VE)
df_V1_result_VE.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [92]:
file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_resimplify_r' + round_idx + '.parquet'

# file_df_V2_result_ET = directory + '/' + tin_filename + '_df_V2_result_ET_resimplify_r11' + '.parquet'
df_V2_result_ET = spark.read.parquet(file_df_V2_result_ET)
df_V2_result_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- subgraphs: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: integer
 |    |    |-- value: integer (valueContainsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [93]:
file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r' + round_idx + '.parquet'

# file_result_con_VE = directory + '/' + tin_filename + '_result_con_VE_resimplify_r11' + '.parquet'
result_con_VE = spark.read.parquet(file_result_con_VE)
result_con_VE.printSchema()

root
 |-- id: integer (nullable = true)
 |-- component: long (nullable = true)



In [94]:
file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r' + round_idx + '.parquet'

# file_result_con_ET = directory + '/' + tin_filename + '_result_con_ET_resimplify_r11' + '.parquet'
result_con_ET = spark.read.parquet(file_result_con_ET)
result_con_ET.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- component: long (nullable = true)



In [95]:
file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r' + round_idx + '.parquet'

# file_df_crit_E_Co_tris = directory + '/' + tin_filename + '_df_crit_E_Co_tris_resimplify_r11' + '.parquet'
df_crit_E_Co_tris = spark.read.parquet(file_df_crit_E_Co_tris)
df_crit_E_Co_tris.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- Saddle_Co_t: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [96]:
# broadcast left join connected components with saddles
saddles_per_con_ET_init = result_con_ET.join(df_crit_E_Co_tris,result_con_ET.tri==df_crit_E_Co_tris.Saddle_Co_t, "inner")

saddles_per_con_ET = saddles_per_con_ET_init.groupBy('component').agg(collect_list('Saddle_edge').alias('multi_Saddles'), collect_list('id').alias('multi_Saddles_co_tri_id'))
saddles_per_con_ET.printSchema()

root
 |-- component: long (nullable = true)
 |-- multi_Saddles: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### update V1-paths and V2-paths

#### reverse part of V1-paths

In [97]:
DF_saddle_minima_True = DF_saddle_minima.filter(DF_saddle_minima.contract_VE_or_not == True)
DF_saddle_minima_True.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_minimum: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_VE_or_not: boolean (nullable = true)



##### jion DF_saddle_minima_True and df_V1_result_VE to reverse part of V1-paths

In [98]:
df_V1_result_VE_update_init = df_V1_result_VE.join(DF_saddle_minima_True, df_V1_result_VE.component == DF_saddle_minima_True.pot_minimum, "left")
df_V1_result_VE_update_init = df_V1_result_VE_update_init.select("component", "pot_saddle", "V1_paths", "multi_Saddles_pts")
df_V1_result_VE_update_init.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [99]:
# a function to reverse the Forman gradient (V1-paths)
def reverse_V1_VE(pot_saddle, V1_paths):
    if not V1_paths:
        return None
    if pot_saddle: # the V1-paths along this component need to be reversed
        index_to_reverse = -1
#         for i in range(len(V1_paths)):
#             if V1_paths[i][0] in pot_saddle:
#                 index_to_reverse = i
#                 # get the end point of  this saddle and this end point is not contained in this component
#                 if V1_paths[i][0] == pot_saddle[0]:
#                     pot_saddle_unique_ver = pot_saddle[1]
#                 else:
#                     pot_saddle_unique_ver = pot_saddle[0]
#                 break
        sad_pt0_connected_with_crit_v = False
        sad_pt1_connected_with_crit_v = False
        for i in range(len(V1_paths)):
            if V1_paths[i] and V1_paths[i][0] in pot_saddle:
                index_to_reverse = i                
                # get the end point of  this saddle and this end point is not contained in this component
                if V1_paths[i][0] == pot_saddle[0]:
                    pot_saddle_unique_ver = pot_saddle[1]
                    sad_pt0_connected_with_crit_v = True
                if V1_paths[i][0] == pot_saddle[1]:
                    pot_saddle_unique_ver = pot_saddle[0]
                    sad_pt1_connected_with_crit_v = True
        
#         if sad_pt0_connected_with_crit_v and sad_pt1_connected_with_crit_v:
#             # both of the two extreme vertices of a saddle are connected with crit_v
#             return None
                
        if index_to_reverse != -1:
            # concatenate the V-paths
            new_V1_paths = []
            set_reverse = set(V1_paths[index_to_reverse])
            for i in range(len(V1_paths)):
                if i != index_to_reverse and V1_paths[i]:
                    # test if part of V1_paths[index_to_reverse] is already in V1_paths[i]
                    set1 = set(V1_paths[i])
                    common_ver = set1.intersection(set_reverse)
                    lst1 = V1_paths[i][0:(len(V1_paths[i]) - len(common_ver) + 1)]
                    lst2 = V1_paths[index_to_reverse][0:(len(V1_paths[index_to_reverse]) - len(common_ver))]
                    lst2.reverse()
                    if pot_saddle_unique_ver and pot_saddle_unique_ver != -1:
                        lst2.append(pot_saddle_unique_ver) # add the other end point of this saddle
                    new_V1_paths.append(lst1+lst2)
                    
                
            if len(new_V1_paths) == 0: # a special case when there is only one path in V2_paths
                V1_paths_origin = V1_paths[index_to_reverse]
                V1_paths_origin.reverse()
                if pot_saddle_unique_ver and pot_saddle_unique_ver != -1:
                    V1_paths_origin.append(pot_saddle_unique_ver)
                    
                new_V1_paths.append(V1_paths_origin)
                
            return new_V1_paths
        
    else: # do not reverse the V1-paths, but we filter out the V1-path with only one node
        new_V1_paths = []
        if V1_paths:
            for i in range(len(V1_paths)):
                if V1_paths[i] and len(V1_paths[i]) > 1:
                    new_V1_paths.append(V1_paths[i])
        return new_V1_paths

reverse_V1_VE_udf = udf(reverse_V1_VE, ArrayType(ArrayType(IntegerType())))

In [100]:
df_V1_result_VE_update = df_V1_result_VE_update_init.withColumn("reversed_V1_paths", reverse_V1_VE_udf(df_V1_result_VE_update_init.pot_saddle, df_V1_result_VE_update_init.V1_paths))
df_V1_result_VE_update.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_pts: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reversed_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [101]:
t0 = time.time()

file_df_V1_result_VE_update = directory + '/' + tin_filename + '_df_V1_result_VE_update_resimplify_r' + round_idx + '.parquet'

# file_df_V1_result_VE_update = directory + '/' + tin_filename + '_df_V1_result_VE_update_resimplify_r11' + '.parquet'
df_V1_result_VE_update.write.parquet(file_df_V1_result_VE_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 2.0200443267822266


#### reverse part of V2-paths

In [102]:
DF_maxima_saddle_True = DF_maxima_saddle.filter(DF_maxima_saddle.contract_ET_or_not == True)
DF_maxima_saddle_True.printSchema()

root
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_maximum_id: long (nullable = true)
 |-- multi_components_VE: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- multi_Max_tri: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- contract_ET_or_not: boolean (nullable = true)



##### jion DF_saddle_maxima_True and df_V2_result_ET to reverse part of V2-paths

In [103]:
df_V2_result_ET_update_init = df_V2_result_ET.join(DF_maxima_saddle_True, df_V2_result_ET.component==DF_maxima_saddle_True.pot_maximum_id, "left")
df_V2_result_ET_update_init = df_V2_result_ET_update_init.select("component", "pot_maximum", "pot_saddle", "V2_paths", "multi_Saddles_co_tri_id")
df_V2_result_ET_update_init.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the triangle id of the co-boundary triangles of pot-saddles

In [104]:
saddles_per_con_ET_init_Co_t = saddles_per_con_ET_init.groupby("Saddle_edge").agg(collect_list('id').alias('two_Saddle_Co_t_id'))
saddles_per_con_ET_init_Co_t.printSchema()

root
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [105]:
df_V2_result_ET_update_init_2 = df_V2_result_ET_update_init.join(saddles_per_con_ET_init_Co_t, df_V2_result_ET_update_init.pot_saddle==saddles_per_con_ET_init_Co_t.Saddle_edge, "left")
df_V2_result_ET_update_init_2 = df_V2_result_ET_update_init_2.select("component", "pot_maximum", "pot_saddle", "two_Saddle_Co_t_id", "V2_paths", "multi_Saddles_co_tri_id")
df_V2_result_ET_update_init_2.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [106]:
# a function to reverse the Forman gradient (V1-paths)
def reverse_V2_ET(two_Saddle_Co_t_id, V2_paths):
    if two_Saddle_Co_t_id:
        index_to_reverse = -1
#         for i in range(len(V2_paths)):
#             if V2_paths[i][0] in two_Saddle_Co_t_id:
#                 index_to_reverse = i
#                 # get the co-boundary triangle this saddle and this co-boundary triangle is not contained in this component
#                 if V2_paths[i][0] == two_Saddle_Co_t_id[0]:
#                     pot_saddle_unique_co_t = two_Saddle_Co_t_id[1]
#                 else:
#                     pot_saddle_unique_co_t = two_Saddle_Co_t_id[0]
#                 break
        
        sad_Co_t0_connected_with_crit_t = False
        sad_Co_t1_connected_with_crit_t = False
        pot_saddle_unique_co_t = None
        for i in range(len(V2_paths)):
            if V2_paths[i][0] in two_Saddle_Co_t_id:
                index_to_reverse = i
#                 if len(two_Saddle_Co_t_id) < 2:
#                     break
                # get the co-boundary triangle this saddle and this co-boundary triangle is not contained in this component
                if len(two_Saddle_Co_t_id) > 1 and V2_paths[i][0] == two_Saddle_Co_t_id[0]:
                    pot_saddle_unique_co_t = two_Saddle_Co_t_id[1]
                    sad_Co_t0_connected_with_crit_t = True
                if len(two_Saddle_Co_t_id) > 1 and V2_paths[i][0] == two_Saddle_Co_t_id[1]:
                    pot_saddle_unique_co_t = two_Saddle_Co_t_id[0]
                    sad_Co_t1_connected_with_crit_t = True
        
#         if sad_Co_t0_connected_with_crit_t and sad_Co_t1_connected_with_crit_t:
#             # both of the two extreme vertices of a saddle are connected with crit_v
#             return None
                
        if index_to_reverse != -1:
           #  V2_paths[index_to_reverse].reverse()
            # concatenate the V-paths
            new_V2_paths = []
            set_reverse = set(V2_paths[index_to_reverse])
            for i in range(len(V2_paths)):
                if i != index_to_reverse:
                    # test if part of V2_paths[index_to_reverse] is already in V2_paths[i]
                    set1 = set(V2_paths[i])
                    common_tri_id = set1.intersection(set_reverse)
                    lst1 = V2_paths[i][0:(len(V2_paths[i]) - len(common_tri_id) + 1)]
                    lst2 = V2_paths[index_to_reverse][0:(len(V2_paths[index_to_reverse]) - len(common_tri_id))]
                    lst2.reverse()
                    if pot_saddle_unique_co_t and pot_saddle_unique_co_t != -1: # a boundary critical triangle if it is -1
                        lst2.append(pot_saddle_unique_co_t) # add the other end point of this saddle
                    new_V2_paths.append(lst1+lst2)
                    
            if len(new_V2_paths) == 0: # a special case when there is only one path in V2_paths
                V2_paths_origin = V2_paths[index_to_reverse]
                V2_paths_origin.reverse()
                if pot_saddle_unique_co_t and pot_saddle_unique_co_t != -1:
                    V2_paths_origin.append(pot_saddle_unique_co_t)
                    
                new_V2_paths.append(V2_paths_origin)
                
            return new_V2_paths
    else:
        return V2_paths

reverse_V2_ET_udf = udf(reverse_V2_ET, ArrayType(ArrayType(IntegerType())))

##### reverse V2_paths

In [107]:
df_V2_result_ET_update = df_V2_result_ET_update_init_2.withColumn("reversed_V2_paths", reverse_V2_ET_udf(df_V2_result_ET_update_init_2.two_Saddle_Co_t_id, df_V2_result_ET_update_init_2.V2_paths))
df_V2_result_ET_update.printSchema()

root
 |-- component: long (nullable = true)
 |-- pot_maximum: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- pot_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- multi_Saddles_co_tri_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reversed_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [108]:
t0 = time.time()

file_df_V2_result_ET_update = directory + '/' + tin_filename + '_df_V2_result_ET_update_resimplify_r' + round_idx + '.parquet'

# file_df_V2_result_ET_update = directory + '/' + tin_filename + '_df_V2_result_ET_update_resimplify_r11' + '.parquet'
df_V2_result_ET_update.write.parquet(file_df_V2_result_ET_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 8.107794284820557


### eliminate the involved V1-path and V2-path

#### eliminate minimum-saddle and the involved V2-paths

In [109]:
DF_saddle_minima_remove = DF_saddle_minima_True.select(col("pot_saddle").alias("removed_saddle"), col("pot_minimum").alias("removed_minimum"))
DF_saddle_minima_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_minimum: long (nullable = true)



In [110]:
t0 = time.time()

file_DF_saddle_minima_remove = directory + '/' + tin_filename + '_DF_saddle_minima_remove_resimplify_r' + round_idx + '.parquet'

# file_DF_saddle_minima_remove = directory + '/' + tin_filename + '_DF_saddle_minima_remove_resimplify_r11' + '.parquet'
DF_saddle_minima_remove.write.parquet(file_DF_saddle_minima_remove) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 0.9921576976776123


In [111]:
# get the co-boundary triangles of the removed saddles
DF_saddle_minima_remove_Co_t = DF_saddle_minima_remove.join(saddles_per_con_ET_init_Co_t, DF_saddle_minima_remove.removed_saddle==saddles_per_con_ET_init_Co_t.Saddle_edge, "left")
DF_saddle_minima_remove_Co_t = DF_saddle_minima_remove_Co_t.select("removed_saddle", "removed_minimum","two_Saddle_Co_t_id")
DF_saddle_minima_remove_Co_t.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_minimum: long (nullable = true)
 |-- two_Saddle_Co_t_id: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### get the connected components in G_ET that each co-boundary triangle belongs to

In [112]:
# explode two_Saddle_Co_t_id
DF_saddle_minima_remove_Co_t_expl = DF_saddle_minima_remove_Co_t.select("removed_saddle", "removed_minimum",explode("two_Saddle_Co_t_id").alias("one_Co_t_id"))
DF_saddle_minima_remove_Co_t_expl.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_minimum: long (nullable = true)
 |-- one_Co_t_id: integer (nullable = true)



In [113]:
# get the component that one_Co_t_id belongs to
DF_SadMin_RM_Co_t_expl_com = DF_saddle_minima_remove_Co_t_expl.join(result_con_ET, DF_saddle_minima_remove_Co_t_expl.one_Co_t_id==result_con_ET.id, "left")
DF_SadMin_RM_Co_t_expl_com = DF_SadMin_RM_Co_t_expl_com.select("removed_saddle", "removed_minimum","one_Co_t_id", col("component").alias("one_Co_t_id_component"))
DF_SadMin_RM_Co_t_expl_com.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_minimum: long (nullable = true)
 |-- one_Co_t_id: integer (nullable = true)
 |-- one_Co_t_id_component: long (nullable = true)



In [114]:
DF_SadMin_RM_Co_t_expl_com_gp = DF_SadMin_RM_Co_t_expl_com.groupby("one_Co_t_id_component").agg(collect_list('one_Co_t_id').alias('one_Co_t_ids'))
DF_SadMin_RM_Co_t_expl_com_gp.printSchema()

root
 |-- one_Co_t_id_component: long (nullable = true)
 |-- one_Co_t_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### eliminate the involved V2-paths

In [115]:
df_V2_result_ET_update_rm_init = df_V2_result_ET_update.select("component","reversed_V2_paths")

df_V2_result_ET_update_rm = df_V2_result_ET_update_rm_init.join(DF_SadMin_RM_Co_t_expl_com_gp, df_V2_result_ET_update_rm_init.component==DF_SadMin_RM_Co_t_expl_com_gp.one_Co_t_id_component,"left")
df_V2_result_ET_update_rm = df_V2_result_ET_update_rm.select("component","reversed_V2_paths", "one_Co_t_ids")
df_V2_result_ET_update_rm.printSchema()

root
 |-- component: long (nullable = true)
 |-- reversed_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- one_Co_t_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [116]:
# a function to eliminate the involved V2-paths
def elim_V2_paths(reversed_V2_paths, one_Co_t_ids):
    if one_Co_t_ids and reversed_V2_paths: # this V2-path should be eliminated
        # for each sublist (the V2-path) in reversed_V2_paths, if its first element is not contained in one_Co_t_ids, we eliminate it
        filtered_V2_paths = list(filter(lambda sublist: sublist and sublist[0] not in one_Co_t_ids, reversed_V2_paths))
        return filtered_V2_paths
    else: # return the original V2-paths
        return reversed_V2_paths
    
elim_V2_paths_udf = udf(elim_V2_paths, ArrayType(ArrayType(IntegerType())))

In [117]:
df_V2_result_ET_update_final = df_V2_result_ET_update_rm.withColumn("final_V2_paths", elim_V2_paths_udf(df_V2_result_ET_update_rm.reversed_V2_paths, df_V2_result_ET_update_rm.one_Co_t_ids))
df_V2_result_ET_update_final = df_V2_result_ET_update_final.select("component", "final_V2_paths")
df_V2_result_ET_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V2_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [118]:
t0 = time.time()

file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r' + round_idx + '.parquet'

# file_df_V2_result_ET_update_final = directory + '/' + tin_filename + '_df_V2_result_ET_update_eliminate_resimplify_r11' + '.parquet'
df_V2_result_ET_update_final.write.parquet(file_df_V2_result_ET_update_final) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 7.961456537246704


#### eliminate saddle-maximum and the involved V1-paths

In [119]:
DF_maxima_saddle_remove = DF_maxima_saddle_True.select(col("pot_saddle").alias("removed_saddle"), col("pot_maximum").alias("removed_crti_triangle"))
DF_maxima_saddle_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- removed_crti_triangle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [120]:
t0 = time.time()

file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_resimplify_r' + round_idx + '.parquet'

# file_DF_maxima_saddle_remove = directory + '/' + tin_filename + '_DF_maxima_saddle_remove_resimplify_r11' + '.parquet'
DF_maxima_saddle_remove.write.parquet(file_DF_maxima_saddle_remove) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 0.5049631595611572


##### get the connected components in G_VE that each extreme vertex belongs to

In [121]:
DF_maxima_saddle_remove_End_pt_expl = DF_maxima_saddle_remove.select(explode("removed_saddle").alias("one_End_pt_id"))
DF_maxima_saddle_remove_End_pt_expl.printSchema()

root
 |-- one_End_pt_id: integer (nullable = true)



In [122]:
# get the component that one_End_pt_id belongs to
DF_SadMax_RM_End_pt_expl_com = DF_maxima_saddle_remove_End_pt_expl.join(result_con_VE, DF_maxima_saddle_remove_End_pt_expl.one_End_pt_id==result_con_VE.id, "left")
DF_SadMax_RM_End_pt_expl_com = DF_SadMax_RM_End_pt_expl_com.select("one_End_pt_id", col("component").alias("one_End_pt_id_component"))
DF_SadMax_RM_End_pt_expl_com.printSchema()

root
 |-- one_End_pt_id: integer (nullable = true)
 |-- one_End_pt_id_component: long (nullable = true)



In [123]:
DF_SadMax_RM_End_pt_expl_com_gp = DF_SadMax_RM_End_pt_expl_com.groupby("one_End_pt_id_component").agg(collect_list('one_End_pt_id').alias('one_End_pt_ids'))
DF_SadMax_RM_End_pt_expl_com_gp.printSchema()

root
 |-- one_End_pt_id_component: long (nullable = true)
 |-- one_End_pt_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



##### eliminate the involved V1-paths

In [124]:
df_V1_result_VE_update_rm_init = df_V1_result_VE_update.select("component","reversed_V1_paths")

df_V1_result_VE_update_rm = df_V1_result_VE_update_rm_init.join(DF_SadMax_RM_End_pt_expl_com_gp, df_V1_result_VE_update_rm_init.component==DF_SadMax_RM_End_pt_expl_com_gp.one_End_pt_id_component, "left")
df_V1_result_VE_update_rm = df_V1_result_VE_update_rm.select("component","reversed_V1_paths", "one_End_pt_ids")
df_V1_result_VE_update_rm.printSchema()

root
 |-- component: long (nullable = true)
 |-- reversed_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- one_End_pt_ids: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [125]:
# a function to eliminate the involved V1-paths
def elim_V1_paths(reversed_V1_paths, one_End_pt_ids):
    if one_End_pt_ids and reversed_V1_paths: # this V1-path should be eliminated
        # for each sublist (the V2-path) in reversed_V2_paths, if its first element is not contained in one_Co_t_ids, we eliminate it
        filtered_V1_paths = list(filter(lambda sublist: sublist and sublist[0] not in one_End_pt_ids, reversed_V1_paths))
        return filtered_V1_paths
    else: # return the original V2-paths
        return reversed_V1_paths
    
elim_V1_paths_udf = udf(elim_V1_paths, ArrayType(ArrayType(IntegerType())))

In [126]:
df_V1_result_VE_update_final = df_V1_result_VE_update_rm.withColumn("final_V1_paths", elim_V1_paths_udf(df_V1_result_VE_update_rm.reversed_V1_paths, df_V1_result_VE_update_rm.one_End_pt_ids))
df_V1_result_VE_update_final = df_V1_result_VE_update_final.select("component", "final_V1_paths")
df_V1_result_VE_update_final.printSchema()

root
 |-- component: long (nullable = true)
 |-- final_V1_paths: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)



In [127]:
t0 = time.time()

file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r' + round_idx + '.parquet'

# file_df_V1_result_VE_update_final = directory + '/' + tin_filename + '_df_V1_result_VE_update_eliminate_resimplify_r11' + '.parquet'
df_V1_result_VE_update_final.write.parquet(file_df_V1_result_VE_update_final) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 3.5565507411956787


### Update critical vertices, critical edges, and critical triangles

In [128]:
# _df_crit_V_update_no_threshold is the critical vertices after update but without resimplification
if round_idx == '1':
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update' + '.parquet'
else:
    file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'

# file_df_crit_V = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r10' + '.parquet'
df_crit_V = spark.read.parquet(file_df_crit_V)
df_crit_V.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Min_ver: integer (nullable = true)



In [129]:
if round_idx == '1':
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update' + '.parquet'
else:
    file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_E = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r10' + '.parquet'
df_crit_E = spark.read.parquet(file_df_crit_E)
df_crit_E.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- Saddle_edge: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [130]:
if round_idx == '1':
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update' + '.parquet'
else:
    file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + str(int(round_idx)-1) + '.parquet'
    
# file_df_crit_T = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r10' + '.parquet'
df_crit_T = spark.read.parquet(file_df_crit_T)
df_crit_T.printSchema()

root
 |-- Max_tri: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [131]:
df_crit_V_update = df_crit_V.join(DF_saddle_minima_remove, df_crit_V.Ver==DF_saddle_minima_remove.removed_minimum, "left")

# isNull() function is to check if the column is None (empty). 
df_crit_V_update = df_crit_V_update.filter(col("removed_minimum").isNull()) #The "removed_minimum" column with None means that this crit_V should be kept
df_crit_V_update = df_crit_V_update.select("Ver", "Min_ver")

In [132]:
t0 = time.time()

file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r' + round_idx + '.parquet'

# file_df_crit_V_update = directory + '/' + tin_filename + '_df_crit_V_update_resimplify_r11' + '.parquet'
df_crit_V_update.write.parquet(file_df_crit_V_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.6944215297698975


In [133]:
df_crit_T_update = df_crit_T.join(DF_maxima_saddle_remove, df_crit_T.Max_tri==DF_maxima_saddle_remove.removed_crti_triangle, "left")

df_crit_T_update = df_crit_T_update.filter(col("removed_crti_triangle").isNull())
df_crit_T_update = df_crit_T_update.select("Max_tri")

In [134]:
t0 = time.time()

file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r' + round_idx + '.parquet'

# file_df_crit_T_update = directory + '/' + tin_filename + '_df_crit_T_update_resimplify_r11' + '.parquet'
df_crit_T_update.write.parquet(file_df_crit_T_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.3460969924926758


In [135]:
DF_saddles_remove_VE = DF_saddle_minima_remove.select("removed_saddle")
DF_saddles_remove_ET = DF_maxima_saddle_remove.select("removed_saddle")

DF_saddles_remove = DF_saddles_remove_VE.union(DF_saddles_remove_ET)
DF_saddles_remove.printSchema()

root
 |-- removed_saddle: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [136]:
df_crit_E_update = df_crit_E.join(DF_saddles_remove, df_crit_E.Saddle_edge==DF_saddles_remove.removed_saddle, "left")

df_crit_E_update = df_crit_E_update.filter(col("removed_saddle").isNull())
df_crit_E_update = df_crit_E_update.select("Ver", "Saddle_edge")

In [137]:
t0 = time.time()

file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r' + round_idx + '.parquet'

# file_df_crit_E_update = directory + '/' + tin_filename + '_df_crit_E_update_resimplify_r11' + '.parquet'
df_crit_E_update.write.parquet(file_df_crit_E_update) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 1.4073190689086914


In [138]:
t_whole_program_4 = time.time()
print(f"Total time cost (s): {(t_whole_program_4 - t_whole_program_3)}")
print(f"Total time cost (min): {(t_whole_program_4 - t_whole_program_3) / 60}")

t_step5 = t_whole_program_4 - t_whole_program_3
print(f"Total time cost (min) for step 5: {t_step5 / 60}")

Total time cost (s): 30.197749376296997
Total time cost (min): 0.5032958229382832
Total time cost (min) for step 5: 0.5032958229382832


In [139]:
print(f"Total time cost (min) for step 2: {t_step2 / 60}")
print(f"Total time cost (min) for step 3: {t_step3 / 60}")
print(f"Total time cost (min) for step 4: {t_step4 / 60}")
print(f"Total time cost (min) for step 5: {t_step5 / 60}")

print(f"Total time cost (min) for the whole process: {(t_step2 + t_step3 + t_step4 + t_step5) / 60}")

Total time cost (min) for step 2: 4.0020365436871845
Total time cost (min) for step 3: 3.375445572535197
Total time cost (min) for step 4: 0.15799623727798462
Total time cost (min) for step 5: 0.5032958229382832
Total time cost (min) for the whole process: 8.038774176438649
